# Rental proximity to sport and recreation facilities

This notebook measures how far each rental property is from its nearest
sport and recreation facility. It reads the rental properties and the
cleaned facility coordinates, narrows the facilities for each property
to the nearest few by straight-line distance, and then routes to those
candidates with OpenRouteService.

## Imports and configuration

The path is stored as an upper-case constant, following the PEP 8
convention for module-level constants. Pandas is told to show every
column when a table is displayed, so wide tables are not truncated.

In [86]:
import getpass
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.metrics.pairwise import haversine_distances
from sklearn.neighbors import BallTree

from dotenv import load_dotenv

pd.set_option("display.max_columns", None)

In [87]:
RENTALS_PATH = "../../rea_data/curated/vic_rentals_curated_final.parquet"
FACILITIES_PATH = "../../external_data/sports_rec_facilities_exact.csv"

RENTAL_LAT_COLUMN = "lat"
RENTAL_LON_COLUMN = "lon"
N_CANDIDATES = 10
EARTH_RADIUS_M = 6_371_000

ORS_URL = "https://api.heigit.org/openrouteservice/v2/matrix"  
ORS_PROFILE = "driving-car"
MAX_MATRIX_CELLS = 3_500
MAX_REQUESTS_PER_RUN = 450
SECONDS_BETWEEN_REQUESTS = 1.6
STRIP_DEGREES = 0.05
CACHE_DIR = Path(f"ors_cache_{ORS_PROFILE}")
ROAD_RESULTS_PATH = f"road_distances_{ORS_PROFILE}.csv"

N_CANDIDATES_SECOND_PASS = 50
FINAL_RESULTS_PATH = (
    f"road_distances_final_{ORS_PROFILE}_k{N_CANDIDATES_SECOND_PASS}.csv"
)
SECOND_PASS_CACHE_DIR = Path(
    f"ors_cache_{ORS_PROFILE}_k{N_CANDIDATES_SECOND_PASS}"
)

RENTALS_WITH_PROXIMITY_PATH = (
    f"vic_rentals_with_proximity_{ORS_PROFILE}.parquet"
)

## Read the rental properties


In [88]:
rentals_raw = pd.read_parquet(RENTALS_PATH)

### First look: size, columns and nulls

`info()` lists every column with its data type and the number of
non-null values, and shows the total row count. We are looking for a
property identifier and the latitude and longitude columns, and for how
many of those coordinates are missing.

In [89]:
rentals_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12317 entries, 0 to 12316
Data columns (total 65 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   lat                                 12317 non-null  float64
 1   lon                                 12317 non-null  float64
 2   listing_id                          12317 non-null  object 
 3   suburb                              12317 non-null  object 
 4   weekly_rent                         12317 non-null  float64
 5   date_listed                         12317 non-null  object 
 6   bedrooms                            12317 non-null  float64
 7   bathrooms                           12317 non-null  float64
 8   carspaces                           12317 non-null  float64
 9   property_type                       12317 non-null  object 
 10  feat_built_in_wardrobes             12317 non-null  int32  
 11  feat_heating                        12317

## Read the cleaned facilities

Load the facility ID, latitude and longitude exported by the cleaning
notebook. Every row has exact coordinates, so nothing else is needed.

In [90]:
facilities = pd.read_csv(FACILITIES_PATH)

print(facilities.shape)
facilities.head()

(4909, 3)


,Facility ID,Latitude,Longitude
0,MITCHE3674,-37.851683,145.180228
1,ALPINE5519,-36.737199,146.973353
2,ALPINE10930,-36.740988,147.168502
3,ALPINE10928,-36.739256,147.172162
4,ALPINE10924,-36.475264,147.020525


## Find the unique rental locations

Many rentals share the same coordinates (for example units in one
building), and identical coordinates always have identical distances.
Keep each distinct latitude and longitude pair once, so the distance
work is done per location and not per property. Rows without
coordinates are excluded. The results are joined back to every
property at the end.

The index is reset so that row numbers match the positions in the
nearest-neighbour results below.

In [91]:
rental_coordinates = rentals_raw[
    [RENTAL_LAT_COLUMN, RENTAL_LON_COLUMN]
].dropna()
unique_locations = rental_coordinates.drop_duplicates().reset_index(drop=True)

print(f"{len(rentals_raw)} properties")
print(f"{len(rental_coordinates)} with coordinates")
print(f"{len(unique_locations)} unique locations")

12317 properties
12317 with coordinates
11007 unique locations


## Find the 10 nearest facilities by straight-line distance

A `BallTree` with the haversine metric finds the nearest facilities to
each location without measuring against all of them. Two details:

- Haversine expects coordinates in **radians**, ordered
  **latitude, longitude**, so both tables are converted first.
- Distances come back as angles on the sphere, so multiplying by the
  Earth's radius converts them to metres.

Results are sorted nearest first. `neighbour_positions` holds the row
numbers in `facilities` of the 10 nearest facilities for each location,
and `straight_line_m` holds the matching distances in metres. The last
column of `straight_line_m` is `d10`, the distance to the 10th nearest,
which the later road-distance check depends on.

In [92]:
facility_radians = np.radians(
    facilities[["Latitude", "Longitude"]].to_numpy()
)
location_radians = np.radians(
    unique_locations[[RENTAL_LAT_COLUMN, RENTAL_LON_COLUMN]].to_numpy()
)

tree = BallTree(facility_radians, metric="haversine")
distances_rad, neighbour_positions = tree.query(
    location_radians,
    k=N_CANDIDATES,
)
straight_line_m = distances_rad * EARTH_RADIUS_M

### Check the tree against a brute-force calculation

The most common mistakes here are passing degrees instead of radians
or longitude before latitude. Measure the first 200 locations against
every facility directly and confirm the tree found the same nearest
distance. `True` means the setup is correct.

In [93]:
sample_size = 200
brute_force_m = (
    haversine_distances(location_radians[:sample_size], facility_radians)
    .min(axis=1)
    * EARTH_RADIUS_M
)

# ensure brute force calculations are acceptably close to straight_line_m above
print(np.allclose(brute_force_m, straight_line_m[:sample_size, 0]))

True


### Distribution of straight-line distances

Summarise the distance to the nearest facility and to the 10th nearest
(`d10`), in kilometres. 

In [94]:
distance_summary = pd.DataFrame({
    "nearest_km": straight_line_m[:, 0] / 1000,
    "d10_km": straight_line_m[:, -1] / 1000,
})
distance_summary.describe()

,nearest_km,d10_km
count,11007.000000,11007.000000
mean,0.733067,2.763825
std,0.832874,3.113525
min,0.007979,0.321376
25%,0.317304,1.353407
50%,0.503907,1.754544
75%,0.806493,3.090295
max,21.304374,111.170190


## Plan the batches

Sending one request per location would need 11,007 requests, far more
than the free API allows. Instead, several locations share one matrix
request.

A matrix request returns every source against every destination, so its
size is sources x destinations. Locations that are close together have
mostly the same candidate facilities, so grouping neighbours keeps the
destination list short.

1. **Order the locations.** Cut the map into horizontal strips
   (`STRIP_DEGREES` of latitude tall) and sort by strip, then by
   longitude. Locations that are neighbours end up next to each other.
2. **Pack the batches.** `build_batches` keeps adding locations to the
   current batch until adding one more would push sources x destinations
   over `MAX_MATRIX_CELLS`, then starts a new batch.

In [95]:
strip_number = np.floor(unique_locations[RENTAL_LAT_COLUMN] / STRIP_DEGREES)
processing_order = (
    unique_locations.assign(strip_number=strip_number)
    .sort_values(["strip_number", RENTAL_LON_COLUMN])
    .index.to_numpy()
)


def build_batches(order, candidates, max_cells):
    """Group locations into batches whose ORS matrix fits in max_cells.

    order: location row numbers, arranged so that neighbours are adjacent.
    candidates: array whose row i holds the facility positions for location i.
    Returns a list of (location_rows, facility_positions) pairs.
    """
    batches = []
    sources = []
    destinations = set()

    for location in order:
        location_candidates = set(candidates[location].tolist())
        merged = destinations | location_candidates
        matrix_cells = (len(sources) + 1) * len(merged)

        if sources and matrix_cells > max_cells:
            batches.append((sources, sorted(destinations)))
            sources = []
            merged = location_candidates

        sources.append(int(location))
        destinations = merged

    if sources:
        batches.append((sources, sorted(destinations)))
    return batches

### Dry run: how many requests are needed?

Build the batches and summarise them **before** sending anything, so we
know how much of the daily quota this will use.

- `matrix_cells` is the size of each request (sources x destinations).
- `useful_share` is the fraction of those cells that are actually
  wanted. Each location only needs its own 10 candidates, so the rest of
  its row is wasted. A higher share means fewer requests.

In [96]:
batches = build_batches(
    processing_order,
    neighbour_positions,
    MAX_MATRIX_CELLS,
)

batch_summary = pd.DataFrame({
    "n_locations": [len(rows) for rows, _ in batches],
    "n_facilities": [len(positions) for _, positions in batches],
})
batch_summary["matrix_cells"] = (
    batch_summary["n_locations"] * batch_summary["n_facilities"]
)
useful_pairs = batch_summary["n_locations"] * N_CANDIDATES
batch_summary["useful_share"] = useful_pairs / batch_summary["matrix_cells"]

n_days = int(np.ceil(len(batches) / MAX_REQUESTS_PER_RUN))
print(f"{len(batches)} requests needed, about {n_days} day(s) of quota")
batch_summary.describe()

162 requests needed, about 1 day(s) of quota


,n_locations,n_facilities,matrix_cells,useful_share
count,162.000000,162.000000,162.000000,162.000000
mean,67.944444,55.734568,3399.067901,0.200788
std,24.156387,18.936899,243.347338,0.071918
min,24.000000,19.000000,792.000000,0.083333
25%,53.000000,44.000000,3400.000000,0.158730
50%,64.000000,53.000000,3465.000000,0.188679
75%,77.000000,63.000000,3480.750000,0.227273
max,184.000000,120.000000,3500.000000,0.526316


## Helpers for one ORS request

`request_matrix` sends one batch to the ORS matrix endpoint and returns
two grids, road distances in metres and durations in seconds. ORS
expects coordinates as **longitude, latitude**. `NaN` marks pairs where
no route was found.

`summarise_batch` reduces that grid to one row per location. Each
location is only compared with **its own** 10 candidate facilities
(not every facility in the batch), and the one with the shortest road
distance is kept.

In [97]:
def request_matrix(property_points, facility_points, api_key):
    """Ask ORS for road distances and durations for one batch.

    Both point arrays hold [longitude, latitude] rows. Returns two
    arrays shaped (n_properties, n_facilities), with NaN where ORS
    found no route.
    """
    n_sources = len(property_points)
    n_destinations = len(facility_points)
    body = {
        "locations": np.vstack([property_points, facility_points]).tolist(),
        "sources": list(range(n_sources)),
        "destinations": list(range(n_sources, n_sources + n_destinations)),
        "metrics": ["distance", "duration"],
        "units": "m",
    }
    response = requests.post(
        f"{ORS_URL}/{ORS_PROFILE}",
        json=body,
        headers={"Authorization": api_key},
        timeout=120,
    )
    response.raise_for_status()
    result = response.json()
    distances = np.array(result["distances"], dtype=float)
    durations = np.array(result["durations"], dtype=float)
    return distances, durations


def summarise_batch(
    location_rows,
    facility_positions,
    distances,
    durations,
    candidates,
    facility_ids,
):
    """Reduce a batch's matrix to each location's nearest candidate.

    The matrix holds every location against every facility in the batch,
    but only each location's own candidate facilities are considered.
    """
    column_of = {
        position: column for column, position in enumerate(facility_positions)
    }
    records = []

    for row, location in enumerate(location_rows):
        candidate_positions = candidates[location]
        columns = [column_of[position] for position in candidate_positions]
        candidate_distances = distances[row, columns]
        record = {
            "location_row": location,
            "nearest_facility_id": None,
            "road_distance_m": np.nan,
            "duration_s": np.nan,
        }
        if not np.isnan(candidate_distances).all():
            best = int(np.nanargmin(candidate_distances))
            best_position = candidate_positions[best]
            record["nearest_facility_id"] = facility_ids[best_position]
            record["road_distance_m"] = candidate_distances[best]
            record["duration_s"] = durations[row, columns[best]]
        records.append(record)

    return pd.DataFrame(records)

## Run the batches

`run_pending_batches` goes through the batches in order. For each one it:

1. skips the batch if its result file already exists in the cache
   folder, so a rerun continues where the last run stopped;
2. pauses and stops cleanly once `max_requests` requests have been sent,
   or if ORS answers HTTP 429 (quota reached);
3. sends the request, reduces the result and saves it to its own CSV
   straight away.

The 1.6 second pause between requests keeps us under 40 per minute. The
function reads `unique_locations`, `facilities` and
`neighbour_positions` from the notebook.

In [98]:
def run_pending_batches(
    batches,
    api_key,
    max_requests,
    candidates,
    cache_dir,
):
    """Send each uncached batch to ORS and save its results to cache_dir.

    candidates: array whose row i holds the facility positions for
    location i, the same table that was passed to build_batches.
    """
    cache_dir.mkdir(exist_ok=True)
    facility_ids = facilities["Facility ID"].to_numpy()
    n_sent = 0

    for batch_number, batch in enumerate(batches):
        location_rows, facility_positions = batch
        cache_file = cache_dir / f"batch_{batch_number:04d}.csv"
        if cache_file.exists():
            continue
        if n_sent >= max_requests:
            print(f"Paused after {n_sent} requests. Rerun to continue.")
            return

        property_points = unique_locations.loc[
            location_rows, [RENTAL_LON_COLUMN, RENTAL_LAT_COLUMN]
        ].to_numpy()
        facility_points = facilities.loc[
            facility_positions, ["Longitude", "Latitude"]
        ].to_numpy()

        try:
            distances, durations = request_matrix(
                property_points, facility_points, api_key
            )
        except requests.HTTPError as error:
            status = error.response.status_code
            if status == 429:
                print("ORS quota reached (HTTP 429). Rerun later.")
                return
            print(
                f"HTTP {status} on batch {batch_number} "
                f"({len(location_rows)} x {len(facility_positions)}): "
                f"{error.response.text[:200]}"
            )
            return

        batch_results = summarise_batch(
            location_rows,
            facility_positions,
            distances,
            durations,
            candidates,
            facility_ids,
        )
        batch_results.to_csv(cache_file, index=False)

        n_sent += 1
        if n_sent % 25 == 0:
            print(f"{n_sent} requests sent")
        time.sleep(SECONDS_BETWEEN_REQUESTS)

    print(f"All batches saved ({n_sent} requests sent this run).")

### Send the requests

The API key is read from the `ORS_API_KEY` environment variable, or
typed in when the cell runs, so it is never saved in the notebook file.
Any stray spaces or newlines are stripped, and the key's length is
printed (never the key itself) so an empty or truncated paste is easy to
spot.

This cell can be rerun as often as needed. Batches that are already
saved are skipped. If you change `STRIP_DEGREES`, `MAX_MATRIX_CELLS` or
`N_CANDIDATES`, delete the cache folder first, because the batches will
no longer match the saved files.

In [99]:
load_dotenv()

ORS_API_KEY = os.environ.get("ORS_API_KEY", "").strip()
if not ORS_API_KEY:
    ORS_API_KEY = getpass.getpass("Enter your ORS API key: ").strip()

print(f"Key length: {len(ORS_API_KEY)} characters")

n_saved = len(list(CACHE_DIR.glob("batch_*.csv")))
print(f"{n_saved} of {len(batches)} batches already saved")

run_pending_batches(
    batches,
    ORS_API_KEY,
    MAX_REQUESTS_PER_RUN,
    neighbour_positions,
    CACHE_DIR,
)

Key length: 152 characters
162 of 162 batches already saved
All batches saved (0 requests sent this run).


## Combine the results and apply the check

Read every saved batch into one table and attach it to the unique
locations. `d10_m` is the straight-line distance to each location's
10th-nearest facility. A location is **confirmed** when its best road
distance is at most `d10_m`: any facility outside the 10 candidates is
at least `d10_m` away in a straight line, and a road is never shorter
than a straight line, so nothing outside the candidates can beat it.

Locations that are not confirmed (or have no route at all) need a
second pass with more candidates.

In [100]:
cached_files = sorted(CACHE_DIR.glob("batch_*.csv"))
road_results = pd.concat(
    (pd.read_csv(path) for path in cached_files),
    ignore_index=True,
)
print(f"{len(road_results)} of {len(unique_locations)} locations routed")

location_results = unique_locations.join(
    road_results.set_index("location_row")
)
location_results["d10_m"] = straight_line_m[:, -1]
location_results["is_confirmed"] = (
    location_results["road_distance_m"] <= location_results["d10_m"]
)
location_results.to_csv(ROAD_RESULTS_PATH, index=False)

print(location_results["is_confirmed"].value_counts())

11007 of 11007 locations routed
is_confirmed
True     10648
False      359
Name: count, dtype: int64


## Second pass: more candidates for unconfirmed locations

Locations that failed the check in the first pass get a wider search.

1. **Find them.** Read the saved first-pass results and take the rows
   where `is_confirmed` is false. Reading from the file (not the
   notebook variable) keeps this list fixed if the cell is rerun later.
2. **Widen the candidates.** Ask the tree for the 25 nearest facilities
   to every location (a fast operation) and keep the distance to the
   25th, which becomes the new bound for the check.
3. **Plan the batches** for just the unconfirmed locations, using the
   same strip ordering and the same packing as before, and print how
   many requests that needs.

In [101]:
first_pass = pd.read_csv(ROAD_RESULTS_PATH)
unconfirmed_rows = np.flatnonzero(~first_pass["is_confirmed"].to_numpy(bool))
print(f"{len(unconfirmed_rows)} unconfirmed locations")

wide_distances_rad, wide_positions = tree.query(
    location_radians,
    k=N_CANDIDATES_SECOND_PASS,
)
wide_straight_line_m = wide_distances_rad * EARTH_RADIUS_M

is_unconfirmed = np.isin(processing_order, unconfirmed_rows)
second_pass_order = processing_order[is_unconfirmed]
second_pass_batches = build_batches(
    second_pass_order,
    wide_positions,
    MAX_MATRIX_CELLS,
)
print(f"{len(second_pass_batches)} requests needed")

359 unconfirmed locations
22 requests needed


### Send the second-pass requests

Same runner as before, pointed at the wider candidate table and its own
cache folder, so first-pass results are untouched. Batches that are
already saved are skipped, so the cell can be rerun safely.

In [102]:
run_pending_batches(
    second_pass_batches,
    ORS_API_KEY,
    MAX_REQUESTS_PER_RUN,
    wide_positions,
    SECOND_PASS_CACHE_DIR,
)

All batches saved (0 requests sent this run).


## Merge the second pass and re-check

Start from the saved first-pass table and overwrite the rerouted rows
with their second-pass results. Because the second pass considered a
superset of the first-pass candidates, the new road distance can only be
the same or shorter.

Two audit columns record how each row was confirmed:
`candidates_used` (10 or 25) and `bound_m`, the straight-line distance to
the farthest candidate considered. A location is confirmed when its road
distance is at most `bound_m`. The final table is saved to a new file,
so the first-pass file stays as it was.

In [103]:
cached_files = sorted(SECOND_PASS_CACHE_DIR.glob("batch_*.csv"))
second_pass_results = pd.concat(
    (pd.read_csv(path) for path in cached_files),
    ignore_index=True,
).set_index("location_row")
print(f"{len(second_pass_results)} of {len(unconfirmed_rows)} rerouted")

final_results = first_pass.copy()
final_results["candidates_used"] = N_CANDIDATES
final_results["bound_m"] = final_results["d10_m"]

rerouted_rows = second_pass_results.index.to_numpy()
result_columns = ["nearest_facility_id", "road_distance_m", "duration_s"]
final_results.loc[rerouted_rows, result_columns] = second_pass_results[
    result_columns
]
final_results.loc[rerouted_rows, "candidates_used"] = N_CANDIDATES_SECOND_PASS
final_results.loc[rerouted_rows, "bound_m"] = wide_straight_line_m[
    rerouted_rows, -1
]

final_results["is_confirmed"] = (
    final_results["road_distance_m"] <= final_results["bound_m"]
)
final_results.to_csv(FINAL_RESULTS_PATH, index=False)

print(final_results["is_confirmed"].value_counts())

359 of 359 rerouted
is_confirmed
True     11006
False        1
Name: count, dtype: int64


## Attach the results to every property

The routing was done per unique location, so each result now has to go
back to every property at that location.

1. **Find each property's location row.** `unique_locations` was built
   from the rental coordinates, so merging on latitude and longitude
   gives every property the row number of its location. This is done
   against `unique_locations` (which is in memory) and not the saved
   CSV, because a CSV round trip can change the last digit of a
   coordinate and break an exact match. Properties with no coordinates
   get no location row.
2. **Attach the results** by that row number. Properties without
   coordinates end up with empty distance columns.

A left merge keeps every property, so the row count must stay the same
as the original data. The assertion stops the notebook if it doesn't.

In [104]:
location_lookup = unique_locations.rename_axis("location_row").reset_index()
properties_with_rows = rentals_raw.merge(
    location_lookup,
    on=[RENTAL_LAT_COLUMN, RENTAL_LON_COLUMN],
    how="left",
)
assert len(properties_with_rows) == len(rentals_raw)

result_columns = [
    "nearest_facility_id",
    "road_distance_m",
    "duration_s",
    "is_confirmed",
]
rentals_with_proximity = properties_with_rows.merge(
    final_results[result_columns],
    left_on="location_row",
    right_index=True,
    how="left",
).drop(columns="location_row")
assert len(rentals_with_proximity) == len(rentals_raw)

### Summarise and save

Count how many properties received a distance, how many did not (no
coordinates) and how many are marked unconfirmed. Then summarise the
road distances and save the table as a single parquet file next to the
notebook.

In [105]:
n_properties = len(rentals_with_proximity)
n_with_distance = rentals_with_proximity["road_distance_m"].notna().sum()
n_unconfirmed = rentals_with_proximity["is_confirmed"].eq(False).sum()

print(f"{n_properties} properties")
print(f"{n_with_distance} with a road distance")
print(f"{n_properties - n_with_distance} without (no coordinates)")
print(f"{n_unconfirmed} marked unconfirmed")

rentals_with_proximity.to_parquet(RENTALS_WITH_PROXIMITY_PATH, index=False)
rentals_with_proximity["road_distance_m"].describe()

12317 properties
12317 with a road distance
0 without (no coordinates)
1 marked unconfirmed


count    12317.000000
mean      1125.074603
std       1212.715479
min          0.000000
25%        497.610000
50%        799.350000
75%       1307.810000
max      42200.150000
Name: road_distance_m, dtype: float64

## Settle the last unconfirmed location

One location is still unconfirmed. Route it to **every** facility, in as
many requests as the matrix limit needs (2 for about 4,900 facilities),
and take the shortest road distance. Nothing is left to prove after
that, because no facility was left out.

The result is printed next to the previous one so they can be compared.

In [106]:
unconfirmed_rows = final_results.index[
    final_results["is_confirmed"].eq(False)
].to_numpy()
assert len(unconfirmed_rows) == 1
location_row = int(unconfirmed_rows[0])

property_point = unique_locations.loc[
    [location_row], [RENTAL_LON_COLUMN, RENTAL_LAT_COLUMN]
].to_numpy()

n_chunks = int(np.ceil(len(facilities) / MAX_MATRIX_CELLS))
chunks = np.array_split(np.arange(len(facilities)), n_chunks)

distance_chunks = []
duration_chunks = []
for chunk in chunks:
    facility_points = facilities.loc[
        chunk, ["Longitude", "Latitude"]
    ].to_numpy()
    distances, durations = request_matrix(
        property_point, facility_points, ORS_API_KEY
    )
    distance_chunks.append(distances[0])
    duration_chunks.append(durations[0])
    time.sleep(SECONDS_BETWEEN_REQUESTS)

all_distances = np.concatenate(distance_chunks)
all_durations = np.concatenate(duration_chunks)

best_position = int(np.nanargmin(all_distances))
best_facility_id = facilities["Facility ID"].iloc[best_position]
best_distance_m = all_distances[best_position]
best_duration_s = all_durations[best_position]

previous = final_results.loc[location_row]
previous_id = previous["nearest_facility_id"]
previous_m = previous["road_distance_m"]

print(f"Previous result: {previous_id}, {previous_m:.0f} m")
print(f"Full search: {best_facility_id}, {best_distance_m:.0f} m")

Previous result: YARRAC10968, 3053 m
Full search: YARRAC10967, 3053 m


### Record the full-search result

Write the full-search result into `final_results` for this location and
mark it confirmed. `candidates_used` records that every facility was
checked, and `bound_m` is left empty because no bound is needed when
nothing was excluded.

In [107]:
updated_columns = ["nearest_facility_id", "road_distance_m", "duration_s"]
final_results.loc[location_row, updated_columns] = [
    best_facility_id,
    best_distance_m,
    best_duration_s,
]
final_results.loc[location_row, "candidates_used"] = len(facilities)
final_results.loc[location_row, "bound_m"] = np.nan
final_results.loc[location_row, "is_confirmed"] = True
final_results.to_csv(FINAL_RESULTS_PATH, index=False)